In [2]:
# ==========================================
# OPTIMIZED DEMAND PREDICTION MODEL (Best R²)
# ==========================================
# !pip install pandas numpy xgboost scikit-learn joblib

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
print("--- Step 1: Loading Data ---")
df = pd.read_csv('Crop-Demand-Data.csv')
df['date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))
df = df.sort_values(by=['Crop', 'Region', 'date']).reset_index(drop=True)
print(f"Data loaded. Shape: {df.shape}")

# 2. Advanced Feature Engineering
print("\n--- Step 2: Feature Engineering ---")
df['year'] = df['date'].dt.year
df['month_num'] = df['date'].dt.month

# Lag Features
df['demand_lag_1'] = df.groupby(['Crop', 'Region'])['Market_Demand'].shift(1)
df['demand_lag_2'] = df.groupby(['Crop', 'Region'])['Market_Demand'].shift(2)
df['demand_lag_3'] = df.groupby(['Crop', 'Region'])['Market_Demand'].shift(3)

# Rolling Mean
df['demand_rolling_mean_3'] = df.groupby(['Crop', 'Region'])['Market_Demand'].transform(
    lambda x: x.rolling(window=3).mean()
)

df = df.dropna().reset_index(drop=True)

# Encode categorical variables
le_region = LabelEncoder()
le_crop = LabelEncoder()
df['region_encoded'] = le_region.fit_transform(df['Region'])
df['crop_encoded'] = le_crop.fit_transform(df['Crop'])
print("Feature Engineering complete.")

# 3. Feature Selection & Time-Based Split
print("\n--- Step 3: Data Splitting ---")
features = ['year', 'month_num', 'region_encoded', 'crop_encoded', 
            'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_rolling_mean_3']
target = 'Market_Demand'

X = df[features]
y = df[target]

split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")

# 4. Model Training (Tuned XGBoost)
print("\n--- Step 4: Model Training ---")
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)
print("Model training complete.")

# 5. Evaluation (R² and Threshold Accuracy)
print("\n--- Step 5: Evaluation ---")
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

# Threshold-Based Accuracy: % of predictions within 15% of actual
within_15_percent = np.abs((y_test - predictions) / (y_test + 1e-9)) <= 0.15
accuracy_percentage = np.mean(within_15_percent) * 100

print(f"Demand Prediction Model - MAE: {mae:.2f}")
print(f"--> Demand Prediction R² Score: {r2:.4f} (Closer to 1.0 is better)")
print(f"--> Demand Prediction Overall Accuracy: {accuracy_percentage:.2f}% (Predictions within 15% of actual)")

# 6. Save Model & Encoders
joblib.dump(model, 'demand_model.pkl')
joblib.dump(le_region, 'le_region.pkl')
joblib.dump(le_crop, 'le_crop.pkl')
print("\nDemand Prediction Model and Encoders Saved Successfully.")

--- Step 1: Loading Data ---
Data loaded. Shape: (2280, 6)

--- Step 2: Feature Engineering ---
Feature Engineering complete.

--- Step 3: Data Splitting ---
Training set: (1596, 8), Testing set: (399, 8)

--- Step 4: Model Training ---
Model training complete.

--- Step 5: Evaluation ---
Demand Prediction Model - MAE: 273.68
--> Demand Prediction R² Score: 0.8716 (Closer to 1.0 is better)
--> Demand Prediction Overall Accuracy: 61.65% (Predictions within 15% of actual)

Demand Prediction Model and Encoders Saved Successfully.
